# Entrenamiento QLoRA — Qwen2.5-1.5B-Instruct para YouTube Boost AI

Este notebook entrena el adaptador QLoRA del proyecto con **Qwen/Qwen2.5-1.5B-Instruct**, usando el CSV de **200 ejemplos sintéticos** y exportando el adaptador listo para que la app lo despliegue.

**Ruta final esperada dentro del repositorio:**

```text
models/qwen2_5_1_5b_marketing_qlora/
```

Al finalizar, esa carpeta debe contener, como mínimo:

```text
adapter_config.json
adapter_model.safetensors
tokenizer.json / tokenizer_config.json
training_metadata.json
training_report.md
inference_smoke.md
```

La app debe apuntar a esa ruta con:

```bash
ENABLE_LOCAL_LLM=true
LOCAL_LLM_MODEL=Qwen/Qwen2.5-1.5B-Instruct
LOCAL_LORA_PATH=models/qwen2_5_1_5b_marketing_qlora
```


## 1. Instalar dependencias

Ejecuta esto en Colab/Kaggle con GPU. Para QLoRA real se recomienda GPU T4, L4, A10, A100 o superior.


In [ ]:
!pip -q install -U \
  "transformers>=4.45.0" \
  "accelerate>=0.34.0" \
  "peft>=0.12.0" \
  "trl>=0.11.0" \
  "datasets>=2.20.0" \
  "bitsandbytes>=0.43.3" \
  "safetensors>=0.4.5" \
  "huggingface_hub>=0.24.0" \
  "sentencepiece>=0.2.0"


## 2. Ubicar la raíz del repositorio

Ejecuta el notebook desde la raíz del repo. Si estás en Colab, sube/clona el repositorio y ajusta `REPO_DIR`.


In [ ]:
from pathlib import Path
import os
import json
import random
import math
import csv
import zipfile
import shutil
from datetime import datetime, timezone

# OPCIÓN A: si ya abriste el notebook desde la raíz del repo, deja Path.cwd().
REPO_DIR = Path.cwd()

# OPCIÓN B Colab/Drive: descomenta y ajusta si tu repo está en Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# REPO_DIR = Path('/content/drive/MyDrive/GRUPO_14-main')

os.chdir(REPO_DIR)
print('Raíz del repo:', REPO_DIR.resolve())
print('Existe data/qlora:', (REPO_DIR / 'data' / 'qlora').exists())


## 3. Configuración del entrenamiento

Estos valores están pensados para estabilizar el lenguaje del modelo en español, reducir ambigüedad y mantener respuestas con lectura métrica real. Con 200 ejemplos, el objetivo es adaptar estilo y estructura, no convertir al LLM en un predictor numérico nuevo.


In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT_DIR = REPO_DIR / 'models' / 'qwen2_5_1_5b_marketing_qlora'
CSV_PATH = REPO_DIR / 'data' / 'qlora' / 'marketing_recommendations_es.csv'
TRAIN_FILE = REPO_DIR / 'data' / 'qlora' / 'train.jsonl'
VAL_FILE = REPO_DIR / 'data' / 'qlora' / 'val.jsonl'

# Caché local del notebook. No es obligatorio commitearlo.
os.environ['HF_HOME'] = str(REPO_DIR / '.cache' / 'huggingface')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Hiperparámetros QLoRA
EPOCHS = 4.0
MAX_STEPS = 240          # Usa -1 para entrenar solo por épocas. 240 da una adaptación más fuerte para dataset pequeño.
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = 1.5e-4
MAX_SEQ_LENGTH = 1536
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.03
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
SEED = 42
USE_RSLORA = True
EXPORT_MERGED_MODEL = False  # La app usa adaptador QLoRA; fusionar el modelo completo ocupa mucho más espacio.

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Modelo base:', MODEL_ID)
print('Salida QLoRA:', OUTPUT_DIR)
print('CSV:', CSV_PATH)


## 4. Verificar CSV de 200 ejemplos y reconstruir splits

El CSV debe tener columnas `input` y `output`. Si tu repo todavía tiene el CSV antiguo de 50 filas, reemplázalo primero por el archivo actualizado en `data/qlora/marketing_recommendations_es.csv`.


In [ ]:
SYSTEM_PROMPT = (
    'Eres un analista senior de marketing, contenido audiovisual, pauta digital y evaluación de métricas. '
    'Responde en español natural, profesional y accionable. No uses JSON. '
    'No inventes cifras: interpreta únicamente las métricas recibidas. '
    'Diferencia intención de contenido: si es humor/entretenimiento, branding o educación, no fuerces un CTA de compra. '
    'Entrega veredictos claros, riesgos, lectura métrica y acciones específicas.'
)


def read_csv_rows(path: Path):
    if not path.exists():
        raise FileNotFoundError(f'No existe el CSV: {path}')
    with path.open('r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    return rows


def get_input_output(row: dict):
    inp = row.get('input') or row.get('prompt') or row.get('entrada') or ''
    out = row.get('output') or row.get('response') or row.get('respuesta') or ''
    return inp.strip(), out.strip()


def validate_rows(rows, min_rows=200):
    if len(rows) < min_rows:
        raise ValueError(
            f'El CSV tiene {len(rows)} filas. Para este QLoRA debes usar mínimo {min_rows}. '
            'Reemplaza data/qlora/marketing_recommendations_es.csv por el CSV actualizado de 200 filas.'
        )
    bad = []
    for i, row in enumerate(rows, start=2):
        inp, out = get_input_output(row)
        if not inp or not out:
            bad.append(i)
    if bad:
        raise ValueError(f'Hay filas sin input/output. Primeras filas problemáticas: {bad[:10]}')


def row_to_messages(row: dict):
    inp, out = get_input_output(row)
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': inp},
            {'role': 'assistant', 'content': out},
        ]
    }


def write_jsonl(path: Path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')

rows = read_csv_rows(CSV_PATH)
validate_rows(rows, min_rows=200)
random.Random(SEED).shuffle(rows)

val_ratio = 0.12
n_val = max(1, int(len(rows) * val_ratio))
val_rows = rows[:n_val]
train_rows = rows[n_val:]

write_jsonl(TRAIN_FILE, [row_to_messages(r) for r in train_rows])
write_jsonl(VAL_FILE, [row_to_messages(r) for r in val_rows])

summary = {
    'source_csv': str(CSV_PATH),
    'total_rows': len(rows),
    'train_rows': len(train_rows),
    'val_rows': len(val_rows),
    'val_ratio': val_ratio,
    'seed': SEED,
    'system_prompt': SYSTEM_PROMPT,
}
summary_path = REPO_DIR / 'data' / 'qlora' / 'dataset_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'OK: {len(train_rows)} train / {len(val_rows)} val. Total CSV: {len(rows)}')
print('Resumen:', summary_path)


## 5. Preparar modelo, tokenizer y LoRA

Esta sección carga Qwen2.5-1.5B-Instruct en 4-bit NF4 y aplica LoRA sobre los módulos de atención y MLP del transformer.


In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed

if not torch.cuda.is_available():
    raise RuntimeError('No hay GPU CUDA disponible. En Colab activa: Entorno de ejecución > Cambiar tipo > GPU.')

set_seed(SEED)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('BF16 soportado:', torch.cuda.is_bf16_supported())

use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    torch_dtype=compute_dtype,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_kwargs = dict(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
if USE_RSLORA:
    lora_kwargs['use_rslora'] = True
try:
    peft_config = LoraConfig(**lora_kwargs)
except TypeError:
    lora_kwargs.pop('use_rslora', None)
    peft_config = LoraConfig(**lora_kwargs)

print(peft_config)


## 6. Entrenar QLoRA

El trainer calcula pérdida de entrenamiento y validación. Al final guardará el adaptador y el tokenizer en la carpeta esperada por la app.


In [ ]:
from inspect import signature
from trl import SFTConfig, SFTTrainer


def formatting_func(example):
    return tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)


def safe_perplexity(eval_loss):
    try:
        val = float(eval_loss)
        if not math.isfinite(val) or val > 20:
            return None
        return float(math.exp(val))
    except Exception:
        return None


def make_sft_config():
    params = signature(SFTConfig.__init__).parameters
    kwargs = {
        'output_dir': str(OUTPUT_DIR),
        'num_train_epochs': EPOCHS,
        'max_steps': MAX_STEPS,
        'per_device_train_batch_size': BATCH_SIZE,
        'per_device_eval_batch_size': 1,
        'gradient_accumulation_steps': GRAD_ACCUM,
        'learning_rate': LR,
        'warmup_ratio': WARMUP_RATIO,
        'weight_decay': WEIGHT_DECAY,
        'lr_scheduler_type': 'cosine',
        'optim': 'paged_adamw_8bit',
        'logging_steps': 5,
        'save_strategy': 'steps',
        'save_steps': 40,
        'eval_steps': 40,
        'max_seq_length': MAX_SEQ_LENGTH,
        'packing': False,
        'bf16': use_bf16,
        'fp16': not use_bf16,
        'gradient_checkpointing': True,
        'group_by_length': True,
        'report_to': 'none',
        'seed': SEED,
        'remove_unused_columns': False,
    }
    # TRL/Transformers cambió el nombre entre versiones.
    if 'eval_strategy' in params:
        kwargs['eval_strategy'] = 'steps'
    elif 'evaluation_strategy' in params:
        kwargs['evaluation_strategy'] = 'steps'
    # Filtra argumentos no soportados en la versión instalada.
    filtered = {k: v for k, v in kwargs.items() if k in params}
    return SFTConfig(**filtered)


ds = load_dataset('json', data_files={'train': str(TRAIN_FILE), 'validation': str(VAL_FILE)})
print(ds)

sft_args = make_sft_config()
trainer_kwargs = dict(
    model=model,
    args=sft_args,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    peft_config=peft_config,
    formatting_func=formatting_func,
)
trainer_params = signature(SFTTrainer.__init__).parameters
if 'processing_class' in trainer_params:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_params:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)
train_result = trainer.train()

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

train_metrics = dict(train_result.metrics or {})
eval_metrics = dict(trainer.evaluate() or {})
ppl = safe_perplexity(eval_metrics.get('eval_loss'))
if ppl is not None:
    eval_metrics['perplexity'] = round(ppl, 4)

print('Train metrics:', train_metrics)
print('Eval metrics:', eval_metrics)


## 7. Guardar reporte, metadatos y prueba de inferencia

Esta celda deja evidencia del entrenamiento y una respuesta de prueba para verificar que el modelo ya no hable con ambigüedad.


In [ ]:
def format_metrics_markdown(payload: dict) -> str:
    train = payload.get('train_metrics', {}) or {}
    evalm = payload.get('eval_metrics', {}) or {}
    lines = [
        '# Reporte real de entrenamiento QLoRA',
        '',
        f"- Base model: `{payload.get('base_model')}`",
        f"- Adaptador: `{payload.get('output_dir')}`",
        f"- Dataset: {payload.get('train_rows')} train / {payload.get('val_rows')} validación",
        f"- LoRA: r={payload.get('lora_r')}, alpha={payload.get('lora_alpha')}, dropout={payload.get('lora_dropout')}",
        f"- Max steps: {payload.get('max_steps')} | epochs: {payload.get('epochs')} | max_seq_length: {payload.get('max_seq_length')}",
        '',
        '## Métricas de entrenamiento',
    ]
    for k in ['train_loss', 'train_runtime', 'train_samples_per_second', 'train_steps_per_second', 'total_flos']:
        if k in train:
            lines.append(f'- {k}: {train[k]}')
    lines.extend(['', '## Métricas de validación'])
    if evalm:
        for k in ['eval_loss', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second', 'perplexity']:
            if k in evalm:
                lines.append(f'- {k}: {evalm[k]}')
    else:
        lines.append('- No se ejecutó validación.')
    lines.extend([
        '',
        '## Interpretación metodológica',
        '- `eval_loss` más bajo indica mejor ajuste al estilo esperado; no equivale por sí solo a calidad estratégica.',
        '- `perplexity` se calcula como exp(eval_loss). Úsala para comparar corridas con el mismo dataset.',
        '- La salida final debe validarse con casos reales, revisión humana y guardrails de dominio.',
    ])
    return '\n'.join(lines) + '\n'

metadata = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'base_model': MODEL_ID,
    'adapter_type': 'qlora_lora_nf4',
    'language': 'es',
    'domain': 'marketing_youtube_ads_recommendations',
    'output_dir': str(OUTPUT_DIR),
    'train_file': str(TRAIN_FILE),
    'eval_file': str(VAL_FILE),
    'train_rows': len(ds['train']),
    'val_rows': len(ds['validation']),
    'max_steps': MAX_STEPS,
    'epochs': EPOCHS,
    'max_seq_length': MAX_SEQ_LENGTH,
    'lora_r': LORA_R,
    'lora_alpha': LORA_ALPHA,
    'lora_dropout': LORA_DROPOUT,
    'learning_rate': LR,
    'gradient_accumulation_steps': GRAD_ACCUM,
    'batch_size': BATCH_SIZE,
    'warmup_ratio': WARMUP_RATIO,
    'weight_decay': WEIGHT_DECAY,
    'train_metrics': train_metrics,
    'eval_metrics': eval_metrics,
    'cuda_device': torch.cuda.get_device_name(0),
    'bf16': use_bf16,
}

(OUTPUT_DIR / 'training_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'training_report.md').write_text(format_metrics_markdown(metadata), encoding='utf-8')

smoke_prompt = (
    'Analiza un video de educación. Acción sugerida por el modelo predictivo: AJUSTAR ANTES DE IMPULSAR. '
    'Views=42000, likes=1800, comentarios=120, engagement=4.57%, retención=39%, sentimiento positivo=52%, '
    'neutral=31%, negativo=17%, composición visual=61/100, relevancia OCR=34%, riesgo de políticas=bajo. '
    'Redacta una recomendación ejecutiva sin JSON y con lectura métrica.'
)
messages = [
    {'role': 'system', 'content': 'Eres un analista senior de marketing. No inventes cifras. Responde con veredicto, métricas, riesgos y acciones.'},
    {'role': 'user', 'content': smoke_prompt},
]
model.eval()
device = next(model.parameters()).device
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt', return_dict=True)
inputs = {k: v.to(device) for k, v in inputs.items()}
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=360,
        do_sample=False,
        repetition_penalty=1.12,
        no_repeat_ngram_size=4,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
smoke_text = tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
(OUTPUT_DIR / 'inference_smoke.md').write_text('# Smoke test de inferencia\n\n' + smoke_text + '\n', encoding='utf-8')

print('Adaptador guardado en:', OUTPUT_DIR)
print('Reporte:', OUTPUT_DIR / 'training_report.md')
print('Smoke test:', OUTPUT_DIR / 'inference_smoke.md')
print('\n--- Respuesta smoke test ---\n')
print(smoke_text)


## 8. Exportación final para la app

La app **no necesita** que commitees el modelo base completo. Debes copiar/commitear el adaptador entrenado en:

```text
models/qwen2_5_1_5b_marketing_qlora/
```

El modelo base se descarga desde Hugging Face usando `LOCAL_LLM_MODEL=Qwen/Qwen2.5-1.5B-Instruct`.


In [ ]:
# Verificación mínima de archivos que la app necesita.
required = ['adapter_config.json', 'adapter_model.safetensors']
missing = [name for name in required if not (OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Faltan archivos obligatorios del adaptador: {missing}')

# Escribe una guía dentro de la carpeta exportada.
deployment_txt = f'''
Carpeta que debe quedar dentro del repo para despliegue:

models/qwen2_5_1_5b_marketing_qlora/

Variables .env recomendadas:

ENABLE_LOCAL_LLM=true
LOCAL_LLM_MODEL={MODEL_ID}
LOCAL_LORA_PATH=models/qwen2_5_1_5b_marketing_qlora
LLM_LOCAL_MAX_NEW_TOKENS=512
LLM_LOCAL_TEMPERATURE=0.20

No mezclar con models/qwen_marketing_qlora porque ese adaptador era de Qwen 0.5B.
'''.strip() + '\n'
(OUTPUT_DIR / 'DEPLOYMENT_PATH.txt').write_text(deployment_txt, encoding='utf-8')

# Crea ZIP conservando la ruta exacta models/qwen2_5_1_5b_marketing_qlora/.
zip_path = REPO_DIR / 'qwen2_5_1_5b_marketing_qlora_export.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.rglob('*'):
        if p.is_file():
            zf.write(p, p.relative_to(REPO_DIR))

print('ZIP exportado:', zip_path)
print('\nPara desplegar en tu app, extrae el ZIP en la raíz del repo. Debe quedar así:')
print('models/qwen2_5_1_5b_marketing_qlora/adapter_config.json')
print('models/qwen2_5_1_5b_marketing_qlora/adapter_model.safetensors')
print('\n.env:')
print(deployment_txt)


## 9. Opcional: exportar modelo fusionado

Para tu app **no es necesario** fusionar el modelo completo. El adaptador QLoRA es más liviano y coincide con `LOCAL_LORA_PATH`.

Activa esto solo si quieres un modelo completo independiente y tienes suficiente RAM/VRAM/almacenamiento.


In [ ]:
if EXPORT_MERGED_MODEL:
    merged_dir = OUTPUT_DIR / 'merged_model'
    try:
        merged = trainer.model.merge_and_unload()
        merged.save_pretrained(str(merged_dir), safe_serialization=True)
        tokenizer.save_pretrained(str(merged_dir))
        print('Modelo fusionado exportado en:', merged_dir)
    except Exception as exc:
        print(f'No se pudo exportar modelo fusionado: {type(exc).__name__}: {exc}')
        print('El adaptador QLoRA sí quedó guardado y es el artefacto recomendado para la app.')
else:
    print('EXPORT_MERGED_MODEL=False. Se conserva solo el adaptador QLoRA recomendado para despliegue.')


## 10. Prueba local de carga como lo hará la app

Esta prueba carga el modelo base y el adaptador desde `LOCAL_LORA_PATH`. Si falla aquí, la app también fallará.


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ['ENABLE_LOCAL_LLM'] = 'true'
os.environ['LOCAL_LLM_MODEL'] = MODEL_ID
os.environ['LOCAL_LORA_PATH'] = 'models/qwen2_5_1_5b_marketing_qlora'

# Carga rápida en 4-bit para validar compatibilidad del adaptador.
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    torch_dtype=compute_dtype,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
loaded = PeftModel.from_pretrained(base, str(OUTPUT_DIR))
loaded.eval()
print('OK: el adaptador QLoRA carga correctamente sobre', MODEL_ID)
print('Ruta que usará la app:', os.environ['LOCAL_LORA_PATH'])
